# Experiment 3 — class-imbalance robustness (PD only)

Metric vs **minority-class proportion** (nested minority removal: 0.15 → 0.0025). **One line per method, averaged over all included datasets** (±1 std band). Watch AP_normalized — it is the prevalence-corrected metric, so it isolates the *model* effect from the baseline shift.

Figures → `figures/experiment3/` (wiped on rerun).

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, method_ranking_bars, per_dataset_bars,
    learning_curve, imbalance_curve, metric_boxplots,
    hpo_improvement_bars, runtime_performance_scatter,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment3')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
df = load_summary(SUMMARY_DIR, experiment='experiment3', task='pd')
print(f'{df["method"].nunique()} methods, {df["dataset"].nunique()} datasets, '
      f'{df["sweep_value"].nunique()} sweep points')

In [ ]:
imbalance_curve(df, 'AUC',           task_name='PD', out_dir=FIGURES_DIR)
imbalance_curve(df, 'AP_normalized', task_name='PD', out_dir=FIGURES_DIR)

In [ ]:
imbalance_curve(df, 'KS', task_name='PD', out_dir=FIGURES_DIR)
imbalance_curve(df, 'F1', task_name='PD', out_dir=FIGURES_DIR)

## Degradation: how much AUC does each method lose from the easiest to the hardest setting?

In [ ]:
import pandas as _pd
g = df.groupby(['method','sweep_value'])['metric.AUC_mean'].mean().reset_index()
drop = {m: gg.sort_values('sweep_value').iloc[-1]['metric.AUC_mean']
            - gg.sort_values('sweep_value').iloc[0]['metric.AUC_mean']
        for m, gg in g.groupby('method')}
display(_pd.Series(drop, name='AUC at max minority - AUC at min minority').sort_values(ascending=False))